# Train and segment one area

This notebook trains on a clipped labeled training area and then runs inference on the larger clipped raster extent.

It walks through:

- prepare one labeled training tile
- build SAM training chips
- train a SAM decoder
- inspect SAM proposal outputs on the training area
- build StarDist training records
- train StarDist
- run segmentation on the larger clipped inference area

This is a tutorial for understanding and exercising the full pipeline. It is not a recommended evaluation setup. For real model assessment, keep separate training and held-out areas.


## Inputs

This notebook assumes:

- a clipped training package already exists under `data/training_data_strong/`
- the larger clipped inference rasters already exist in `data/`


In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "samstars").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from samstars.data import build_scale_stats_from_proposal_records, build_single_tile
from samstars.segmentation import run_segmentation
from samstars.training.chips import build_training_chips
from samstars.training.records import build_stardist_training_records
from samstars.training.sam_training import train_decoder
from samstars.training.stardist_training import train_stardist_model


/Users/iosefa/miniforge3/envs/samstars/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


## Set paths

Training uses the clipped package under `data/training_data_strong/`. Inference uses the larger clipped rasters in `data/`.


In [2]:
data_dir = repo_root / "data"
training_data_dir = data_dir / "training_data_strong"

training_img = training_data_dir / "img.tif"
training_chm = training_data_dir / "chm.tif"
training_den = training_data_dir / "den.tif"
training_crowns = training_data_dir / "crowns.gpkg"
training_extent = training_data_dir / "training_extent.geojson"

inference_img = data_dir / "img_clipped.tif"
inference_chm = data_dir / "chm_clipped.tif"
inference_den = data_dir / "den_clipped.tif"

sam_base_ckpt = repo_root / "sam/sam_vit_l_0b3195.pth"
work_dir = data_dir / "train_and_segment_one_area_run"
device = "mps"

training_tiles_dir = work_dir / "training_labeled_tiles"
training_chips_dir = work_dir / "training_chips"
training_run_dir = work_dir / "training_run"
records_dir = work_dir / "stardist_training_records"
models_dir = work_dir / "models"
inference_run_dir = work_dir / "segmentation_run"
sam_ckpt = models_dir / "sam_crowns_decoder.pth"
stardist_model = models_dir / "stardist"

print("repo_root:", repo_root)
print("training_data_dir:", training_data_dir)
print("inference_img:", inference_img)


repo_root: /Users/iosefa/repos/sam
training_data_dir: /Users/iosefa/repos/sam/data/training_data_strong
inference_img: /Users/iosefa/repos/sam/data/img_clipped.tif


## Set tutorial parameters

These are explicit on purpose. Do not treat them as fixed project defaults. Change them, rerun the notebook, and compare the intermediate proposal outputs before committing to StarDist training.

The seed settings below mirror the larger workflow seed generation:

- `d_min = 4.0`
- `min_dist_px = 4`
- `gauss_sigma = 1`
- `min_samples = 1`
- `merge_radius = 3.0`
- `min_canopy_height = 2.0`
- thinning off


In [3]:
rgb_bands = (1, 2, 3)

seed_d_min = 4.0
seed_min_dist_px = 4
seed_gauss_sigma = 1
seed_min_samples = 1
seed_merge_radius = 3.0
seed_min_canopy_height = 2.0
thin_seeds = False

chip_size = 512
chip_stride = 384

sam_epochs = 3
stardist_epochs = 10
val_fraction = 0.2

crop_smooth_sigma = 2.0
smooth_sigma = 4.0
post_smooth_sigma = 0.0

stardist_input_key = "index"
use_aux = True
stardist_prob_thr = 0.3
stardist_nms_thr = 0.4
stardist_window_size_m = 150.0
stardist_window_overlap_m = 30.0
stardist_dedup_iou_thr = 0.6


## Prepare one labeled training tile

This creates the tile layout used by the downstream training helpers. It rewrites the single prepared training tile so the saved seeds match the same settings as the larger workflow.


In [4]:
build_single_tile(
    img=training_img,
    chm=training_chm,
    den=training_den,
    crowns=training_crowns,
    rgb_bands=rgb_bands,
    out=training_tiles_dir,
    thin_seeds=thin_seeds,
    seed_d_min=seed_d_min,
    seed_min_dist_px=seed_min_dist_px,
    seed_gauss_sigma=seed_gauss_sigma,
    seed_min_samples=seed_min_samples,
    seed_merge_radius=seed_merge_radius,
    seed_min_canopy_height=seed_min_canopy_height,
    skip_existing=False,
)


function_base.py (4824): Warning: 'partition' will ignore the 'mask' of the MaskedArray.
cost_surface.py (137): No SLIC provided – cost built from 3 terms only.
_base.py (248): Precomputed sparse input was not sorted by row values. Use the function sklearn.neighbors.sort_graph_by_row_values to sort the input by row values, with warn_when_not_sorted=False to remove this warning.


✓ Single-tile inputs written to /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/training_labeled_tiles


PosixPath('/Users/iosefa/repos/sam/data/train_and_segment_one_area_run/training_labeled_tiles')

## Build SAM training chips


In [5]:
build_training_chips(
    tiles_dir=training_tiles_dir,
    out_dir=training_chips_dir,
    chip_size=chip_size,
    stride=chip_stride,
)


tile_000_000: wrote 147 training chips
Built 147 training chips in /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/training_chips


147

## Train models

This section first trains the SAM decoder, then creates and inspects proposal rasters on the training area, and only then trains StarDist.


In [6]:
train_decoder(
    chip_dir=training_chips_dir,
    ckpt=sam_base_ckpt,
    out_path=sam_ckpt,
    epochs=sam_epochs,
    device=device,
)


training_chips: epochs=3 (iters/epoch=147)
training_chips epoch 1/3 loss=0.4734
training_chips epoch 2/3 loss=0.3074
training_chips epoch 3/3 loss=0.2143
✓ weights → /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/models/sam_crowns_decoder.pth


In [7]:
proposal_result = run_segmentation(
    tiles_dir=training_tiles_dir,
    out_dir=training_run_dir,
    sam_ckpt=sam_ckpt,
    device=device,
    run_sam=True,
    run_stardist=False,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
    fast=False,
)


seeds in chip: 486
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 9.27310516e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.21389416e-06 7.62803137e-01
 9.77907479e-01]
prob_s pcts: [0.00000000e+00 1.36440809e-04 3.82209498e-01 7.11972899e-01
 8.66928875e-01]
✓ wrote proposal outputs (1 records)


proposals.py (105): invalid value encountered in divide


## Inspect SAM proposal outputs on the training area

Inspect the proposal rasters before continuing. If the proposals are poor, adjust the parameters above and rerun this section before training StarDist.


In [8]:
proposal_result


{'records_root': PosixPath('/Users/iosefa/repos/sam/data/train_and_segment_one_area_run/training_run/run_records'),
 'proposals_root': PosixPath('/Users/iosefa/repos/sam/data/train_and_segment_one_area_run/training_run/proposals'),
 'crowns_dir': PosixPath('/Users/iosefa/repos/sam/data/train_and_segment_one_area_run/training_run/crowns'),
 'proposal_records': PosixPath('/Users/iosefa/repos/sam/data/train_and_segment_one_area_run/training_run/proposals/proposal_records.json')}

In [9]:
proposal_rasters_dir = proposal_result["proposals_root"] / "rasters"
sorted(p.name for p in proposal_rasters_dir.glob("*.tif"))[:20]


['tile_000_000_0_chm.tif',
 'tile_000_000_0_cost.tif',
 'tile_000_000_0_logit.tif',
 'tile_000_000_0_sam_index.tif',
 'tile_000_000_0_sam_prob.tif']

## Build StarDist training records


In [10]:
proposal_result = {
    "records_root": training_run_dir / "run_records",
    "proposals_root": training_run_dir / "proposals",
    "crowns_dir": training_run_dir / "crowns",
    "proposal_records": training_run_dir / "proposals" / "proposal_records.json",
}

train_records, val_records = build_stardist_training_records(
    proposal_records=proposal_result["proposal_records"],
    tiles_dir=training_tiles_dir,
    out_dir=records_dir,
    val_fraction=val_fraction,
)

print("train records:", len(__import__('json').load(open(train_records))))
print("val records:", len(__import__('json').load(open(val_records))))


✓ wrote /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/stardist_training_records/train_records.json (1 records)
✓ wrote /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/stardist_training_records/val_records.json (0 records)
train records: 1
val records: 0


In [11]:
scale_stats = work_dir / "training_scale_stats.json"

build_scale_stats_from_proposal_records(
    proposal_records=proposal_result["proposal_records"],
    out=scale_stats,
    percentiles=(2.0, 98.0),
    chm_key="chm",
    nodata_values=(-9999.0,),
)


{'percentiles': [2.0, 98.0],
 'chm_01': {'lo': 0.0, 'hi': 8.951163768768312},
 'source_proposal_records': '/Users/iosefa/repos/sam/data/train_and_segment_one_area_run/training_run/proposals/proposal_records.json',
 'records_total': 1,
 'records_missing_or_unreadable_chm': 0}

In [12]:
train_stardist_model(
    train_records=train_records,
    val_records=val_records,
    out_dir=stardist_model.parent,
    model_name=stardist_model.name,
    epochs=stardist_epochs,
    input_key=stardist_input_key,
    use_aux=use_aux,
    scale_stats_path=scale_stats,
)


2026-04-20 12:04:33.499584: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Max
2026-04-20 12:04:33.499657: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-04-20 12:04:33.499660: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 48.00 GB
2026-04-20 12:04:33.499670: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-20 12:04:33.499678: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


epochs=10 (iters/epoch=1)
Using default values: prob_thresh=0.5, nms_thresh=0.4.
no validation records found; reusing training data as validation_data
Epoch 1/10


2026-04-20 12:04:34.680984: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


100/100 ━━━━━━━━━━━━━━━━━━━━ 27s 238ms/step - dist_dist_iou_metric: 0.2416 - dist_loss: 6.5594 - dist_relevant_mae: 6.5588 - dist_relevant_mse: 70.5171 - loss: 1.7048 - prob_kld: 0.2434 - prob_loss: 0.3929 - val_dist_dist_iou_metric: 0.4016 - val_dist_loss: 4.8511 - val_dist_relevant_mae: 4.8501 - val_dist_relevant_mse: 37.1814 - val_loss: 1.1923 - val_prob_kld: 0.0760 - val_prob_loss: 0.2221 - learning_rate: 3.0000e-04
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 23s 232ms/step - dist_dist_iou_metric: 0.3899 - dist_loss: 4.7555 - dist_relevant_mae: 4.7547 - dist_relevant_mse: 36.8860 - loss: 1.1770 - prob_kld: 0.0771 - prob_loss: 0.2260 - val_dist_dist_iou_metric: 0.3706 - val_dist_loss: 4.7177 - val_dist_relevant_mae: 4.7170 - val_dist_relevant_mse: 37.4188 - val_loss: 1.1416 - val_prob_kld: 0.0520 - val_prob_loss: 0.1980 - learning_rate: 3.0000e-04
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 23s 233ms/step - dist_dist_iou_metric: 0.4379 - dist_loss: 4.2110 - dist_relevant_mae: 4.2104 - dist_

## Run segmentation on the larger clipped inference area

This runs the trained SAM and StarDist models on the larger clipped rasters, not the training extent.


In [13]:
import importlib
import samstars.segmentation.engine as seg_engine

importlib.reload(seg_engine)
run_segmentation = seg_engine.run_segmentation

result = run_segmentation(
    img=inference_img,
    chm=inference_chm,
    den=inference_den,
    rgb_bands=rgb_bands,
    out_dir=inference_run_dir,
    sam_ckpt=sam_ckpt,
    stardist_model=stardist_model,
    stardist_input_key=stardist_input_key,
    stardist_prob_thr=stardist_prob_thr,
    stardist_nms_thr=stardist_nms_thr,
    stardist_window_size_m=stardist_window_size_m,
    stardist_window_overlap_m=stardist_window_overlap_m,
    stardist_dedup_iou_thr=stardist_dedup_iou_thr,
    device=device,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
    fast=False,
)


function_base.py (4824): Warning: 'partition' will ignore the 'mask' of the MaskedArray.
cost_surface.py (137): No SLIC provided – cost built from 3 terms only.
_base.py (248): Precomputed sparse input was not sorted by row values. Use the function sklearn.neighbors.sort_graph_by_row_values to sort the input by row values, with warn_when_not_sorted=False to remove this warning.


✓ Single-tile inputs written to /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/segmentation_run/.prepared_tiles/9950156a8081
seeds in chip: 5395
seed prob pcts: [1.58057983e-10 3.09869989e-07 3.45043145e-06 1.12658901e-01
 8.46439540e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.11557551e-06 8.59871104e-01
 9.99737620e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.34487663e-06 9.15049553e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.11557551e-06 9.23248868e-01
 9.99992311e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 7.98611476e-01
 9.99937236e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.21389416e-06 7.72831640e-01
 9.94128406e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 9.16991355e-01
 9.99992788e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06623238e-06 8.90943898e-01
 9.99049783e-01]
seed prob pcts: [1.16361983e-10 1.89327835e-07 3.06074980e-06 8.94966006e-01
 9.99917448e-

proposals.py (105): invalid value encountered in divide


prob_s pcts: [0.00000000e+00 4.36202504e-06 2.71637860e-01 6.24582731e-01
 8.49553347e-01]
✓ wrote proposal outputs (1 records)


bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
2026-04-20 15:06:46.660959: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Max
2026-04-20 15:06:46.660990: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-04-20 15:06:46.660994: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 48.00 GB
2026-04-20 15:06:46.661028: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-20 15:06:46.661037: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> p

Loading network weights from 'weights_best.h5'.
Couldn't load thresholds from 'thresholds.json', using default values. (Call 'optimize_thresholds' to change that.)
Using default values: prob_thresh=0.5, nms_thresh=0.4.
Using effective thresholds: prob_thresh=0.3, nms_thresh=0.4.
✓ wrote 3256 crown polygons to /Users/iosefa/repos/sam/data/train_and_segment_one_area_run/segmentation_run/crowns/crowns.gpkg


## Final output


In [14]:
inference_run_dir / "crowns" / "crowns.gpkg"


PosixPath('/Users/iosefa/repos/sam/data/train_and_segment_one_area_run/segmentation_run/crowns/crowns.gpkg')